# SentiNet: train and benchmark on real datasets (Kaggle)

This notebook trains the SentiNet world model on the **real datasets attached to this notebook** (CIC-IDS2017/2018,
CTU-13, UNSW-NB15, in any combination). It then benchmarks the model against the logistic-regression baseline and the
shuffled-history ablation, and packs everything into one zip.

**Before you press "Run All":**
1. **Add Input** → attach your datasets (for example CIC-IDS2017 and CTU-13).
   - CIC-IDS2017 must be the version **with Timestamp and IP columns** ("GeneratedLabelledFlows" / "TrafficLabelling").
     The "MachineLearningCVE" CSVs have no timestamps and are skipped automatically.
   - CTU-13: the `*.binetflow` files, or CSVs with the same columns.
2. **Settings → Internet: ON** (to download the SentiNet code from GitHub).
3. Accelerator: not needed. **CPU is fine**, because the model is small.
4. **Run All.** It takes about 20–40 minutes per dataset.

**When it finishes:** download `sentinet_kaggle_outputs.zip` from the Output panel (right side → /kaggle/working), and copy
the text block printed by the **last cell**. Send both back.

In [ ]:
# 1. Get the SentiNet code (GitHub; or a Kaggle dataset containing the repo if Internet is off)
import os, sys, glob, subprocess, json, time
WORK = "/kaggle/working"
CODE = f"{WORK}/SentiNet"
if not os.path.exists(f"{CODE}/sentinet/__init__.py"):
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Daryl-69/SentiNet.git", CODE])
    if r.returncode != 0:
        hits = glob.glob("/kaggle/input/**/sentinet/__init__.py", recursive=True)
        assert hits, "No internet and no SentiNet code found in /kaggle/input. Turn Internet ON (Settings) and re-run."
        CODE = os.path.dirname(os.path.dirname(hits[0]))
sys.path.insert(0, CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scapy"], check=False)   # only needed for .pcapng

import numpy as np, pandas as pd, torch
torch.set_num_threads(max(1, os.cpu_count() or 1))
import sentinet
from sentinet.realdata import discover, prepare, split
from sentinet.train import train
from sentinet.engine import Forecaster
from sentinet.benchmark import predict_all, run_benchmark, to_markdown, save_results
print("SentiNet", sentinet.__version__, "| code:", CODE, "| torch", torch.__version__, "| pandas", pd.__version__,
      "| CPUs", os.cpu_count())

In [ ]:
# 2. Settings (the defaults are what we use for the report)
INPUT_ROOT  = "/kaggle/input"
CHUNK_HOURS = 2.0     # each capture is cut into 2-hour chunks; whole chunks go to train / val / test
WINDOW_S    = 60      # one network state per minute
HORIZON_K   = 10      # forecast horizon (windows)
EPOCHS      = 14
STEPS       = 250     # steps per epoch
SEED        = 0
OUT = f"{WORK}/sentinet_out"
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 3. Find the dataset files and detect their format
files = discover(INPUT_ROOT)
usable = [f for f in files if f["format"]]
# group = (Kaggle dataset folder, format) so e.g. CIC-IDS2017 and CTU-13 get separate models
groups = {}
for f in usable:
    ds = f["path"].split("/")[3] if f["path"].startswith("/kaggle/input/") else "data"
    key = f"{ds}__{f['format']}"
    groups.setdefault(key, []).append(f["path"])
print("\nDatasets found:")
for k, v in groups.items():
    print(f"  {k:60s} {len(v)} file(s)")
assert groups, "No usable files found - check the Add Input step and the notes at the top."
json.dump({"files": files, "groups": groups}, open(f"{OUT}/discovered.json", "w"), indent=1)

In [ ]:
# 4. Convert each dataset to the canonical format and cut it into time chunks
profiles = {}
for key, paths in groups.items():
    print(f"\n=== {key} ===")
    t = time.time()
    chunks = prepare(paths, f"{WORK}/prepared/{key}", chunk_hours=CHUNK_HOURS, window=WINDOW_S)
    profiles[key] = chunks
    n_inf = sum(c["has_infiltration"] for c in chunks)
    print(f"-> {len(chunks)} chunks, {n_inf} contain infiltration (Initial Access / Lateral / C2 / Exfil)  [{time.time()-t:.0f}s]")
json.dump(profiles, open(f"{OUT}/chunks.json", "w"), indent=1)

In [ ]:
# 5. Split whole chunks into train / validation / test (stratified by 'contains infiltration')
splits = {}
for key, chunks in profiles.items():
    if len(chunks) < 3 or sum(c["has_infiltration"] for c in chunks) < 2:
        print(f"SKIP {key}: needs >= 3 chunks and >= 2 with infiltration (has {len(chunks)} / "
              f"{sum(c['has_infiltration'] for c in chunks)}). Lower CHUNK_HOURS or add files.")
        continue
    splits[key] = split(chunks, test=0.2, val=0.15, seed=SEED)
    s = splits[key]
    inf = {c["file"]: c["has_infiltration"] for c in chunks}
    print(f"{key}: " + ", ".join(f"{k}={len(v)} ({sum(inf[f] for f in v)} with infiltration)" for k, v in s.items()))
json.dump(splits, open(f"{OUT}/splits.json", "w"), indent=1)

In [ ]:
# 6. Train one world model per dataset (+ logistic-regression baseline + flow scorer)
trained = {}
for key, s in splits.items():
    print(f"\n==================== training on {key} ====================")
    wdir = f"{OUT}/weights_{key}"
    t = time.time()
    train(s["train"], wdir, val_paths=s["val"], epochs=EPOCHS, steps_per_epoch=STEPS,
          window=WINDOW_S, horizon=HORIZON_K, seed=SEED)
    trained[key] = wdir
    print(f"trained in {(time.time()-t)/60:.1f} min -> {wdir}")

In [ ]:
# 7. Benchmark on the held-out test chunks: world model vs logistic regression vs shuffled history
results = []
for key, wdir in trained.items():
    fc = Forecaster.load(wdir)
    t = time.time()
    preds = predict_all(fc, splits[key]["test"], samples=32)
    res = run_benchmark(fc, None, preds=preds, name=f"{key} - held-out test chunks")
    res["train_chunks"], res["val_chunks"], res["test_chunks"] = (len(splits[key][k]) for k in ("train", "val", "test"))
    results.append(res)
    print(to_markdown([res]), f"({time.time()-t:.0f}s)")
# cross-dataset generalisation (only if two or more datasets were trained)
keys = list(trained)
for a in keys:
    for b in keys:
        if a != b:
            fc = Forecaster.load(trained[a])
            res = run_benchmark(fc, None, preds=predict_all(fc, splits[b]["test"], samples=32),
                                name=f"CROSS-DATASET: trained on {a}, tested on {b}")
            results.append(res)
            print(to_markdown([res]))
save_results(results, f"{OUT}/results")

In [ ]:
# 8. Plots: validation AUC per epoch, and one test chunk's forecast timeline vs ground truth
import matplotlib.pyplot as plt
for key, wdir in trained.items():
    meta = json.load(open(f"{wdir}/meta.json"))
    h = meta["history"]
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
    ax[0].plot([e["epoch"] for e in h], [e["val_auc"] for e in h], marker="o")
    ax[0].set_title(f"{key}: validation AUC per epoch"); ax[0].set_ylim(0.5, 1)
    fc = Forecaster.load(wdir)
    test = splits[key]["test"]
    best = max(test, key=lambda f: next(c for c in profiles[key] if c["file"] == f)["has_infiltration"])
    res = fc.run(best)
    tb = res.table
    ax[1].plot(tb["time"], tb["p_infiltration"], label="world model")
    if "p_logreg_baseline" in tb: ax[1].plot(tb["time"], tb["p_logreg_baseline"], ":", label="logistic regression")
    if "true_infiltration_within_K" in tb: ax[1].fill_between(tb["time"], 0, tb["true_infiltration_within_K"].fillna(0), alpha=.2, label="truth: infiltration within K")
    ax[1].axhline(fc.threshold, ls="--", c="k", lw=.8); ax[1].legend(fontsize=8); ax[1].set_title(os.path.basename(best))
    plt.tight_layout(); plt.savefig(f"{OUT}/plot_{key}.png", dpi=110); plt.show()

In [ ]:
# 9. Pack everything and print the summary to send back
import shutil
shutil.make_archive(f"{WORK}/sentinet_kaggle_outputs", "zip", OUT)
print("Download: /kaggle/working/sentinet_kaggle_outputs.zip  (",
      round(os.path.getsize(f"{WORK}/sentinet_kaggle_outputs.zip") / 1e6, 1), "MB )\n")
print("=" * 30, "COPY EVERYTHING BELOW AND SEND IT", "=" * 30)
for key, chunks in profiles.items():
    st = {}
    for c in chunks:
        for k, v in c["stages"].items(): st[k] = st.get(k, 0) + v
    print(f"{key}: {len(chunks)} chunks, {sum(c['flows'] for c in chunks):,} flows, stages {st}")
for key, wdir in trained.items():
    m = json.load(open(f"{wdir}/meta.json"))
    print(f"{key}: threshold={m['config']['threshold']:.3f} lr_threshold={m['config']['lr_threshold']:.3f} "
          f"val_auc_by_epoch={[round(e['val_auc'], 3) for e in m['history']]}")
print()
print(open(f"{OUT}/results/benchmark.md").read())